# Task 2: visit quality and integrity review

All rules operate on outcome, GPS features, remarks, photo reuse and other visits, without using surveyed coordinates. Surveyed locations are brought in only afterwards for diagnostics on development addresses. A visit location is not automatically the current borrower's residence.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'extract_evidence.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from extract_evidence import build_evidence
from integrity_analysis import survey_diagnostics, stress_test

out_dir = ROOT / 'results'
out_dir.mkdir(exist_ok=True)
evidence, details = build_evidence(ROOT / 'clean_data', return_details=True)
evidence.to_csv(ROOT / 'evidence.csv', index=False, float_format='%.9g')
pd.DataFrame([details]).to_csv(out_dir / 'uncertainty_fit_summary.csv', index=False)
dev, quality_by_group, pins, uncertainty_quality = survey_diagnostics(evidence)
quality_by_group.to_csv(out_dir / 'visit_quality_by_group.csv', index=False)
pins.to_csv(out_dir / 'visit_pin_diagnostics.csv', index=False)
uncertainty_quality.to_csv(out_dir / 'visit_uncertainty_diagnostics.csv', index=False)
print('Evidence roles:', evidence.evidence_role.value_counts().to_dict())
print('Fit:', details)
print(uncertainty_quality.to_string(index=False))


In [ ]:
print('Endpoint diagnostics on development surveyed addresses:')
print(quality_by_group[quality_by_group.dimension.isin(
    ['evidence_role','photo_reused_cross_address'])].to_string(index=False))
for field in ['contact_only_error_m','extended_error_m']:
    part = pins[pins[field].notna()]
    print(field, 'n=', len(part), 'median evidence error=',
          round(part[field].median(),1), 'median baseline on same subset=',
          round(part.baseline_error_m.median(),1))


In [ ]:
# Include downweighted-but-ineligible rows to display the photo warning.
eligible = dev[dev.weight > 0].copy()
fig, ax = plt.subplots(figsize=(8, 5))
for reused, group in eligible.groupby('photo_reused_cross_address'):
    ax.scatter(group.weight.clip(lower=1e-6), group.endpoint_error_m.clip(lower=1),
               s=25, alpha=.6, label='Reused photo' if reused else 'Other visits')
    ax.set(xscale='log', yscale='log', xlabel='Trust weight (log scale)',
       ylabel='Endpoint error against survey (m, log scale)',
       title='Visit trust versus error, including low-weight photo flags')
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(out_dir / 'weight_vs_error.png', dpi=160)
plt.show()


## Real photo reuse signal

The identical photo hash occurs on different address IDs in the source data. The observed error contrast is descriptive: it does not prove intent or the causal effect of the penalty. The small surveyed reused-photo subgroup must not be treated as a fraud-detection validation set.


In [ ]:
stress, injection, _ = stress_test(ROOT / 'clean_data')
stress.to_csv(out_dir / 'integrity_stress_results.csv', index=False)
injection.to_csv(out_dir / 'synthetic_attack_rows.csv', index=False)
summary = stress.groupby(['scenario','method']).agg(
    affected_addresses=('address_id','nunique'),
    reported_pins=('pin_shift_m','count'),
    abstentions=('abstained','sum'),
    median_pin_shift_m=('pin_shift_m','median'),
    p90_pin_shift_m=('pin_shift_m', lambda s: s.quantile(.9)),
    median_error_after_m=('error_after_m','median')).reset_index()
summary.to_csv(out_dir / 'integrity_stress_summary.csv', index=False)
print(summary.to_string(index=False))


In [ ]:
methods = ['median, high-trust injection',
           'median, recomputed safeguards',
           'independent-agent cluster, may abstain']
scenarios = ['home_checkin','tea_stall','rogue_agent']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
x = np.arange(3); width = .25
for j, method in enumerate(methods):
    part = summary.set_index(['scenario','method'])
    vals = [part.loc[(s,method),'p90_pin_shift_m'] for s in scenarios]
    axes[0].bar(x + (j-1)*width, vals, width, label=method)
axes[0].set_xticks(x, scenarios)
axes[0].set(ylabel='P90 pin shift among reported pins (m)',
            title='Tail shift under synthetic visits')
axes[0].legend(fontsize=7)
mix = summary[summary.method == methods[2]].set_index('scenario').loc[scenarios]
axes[1].bar(scenarios, mix.reported_pins / mix.affected_addresses, color='#0f766e')
axes[1].set(ylim=(0,1), ylabel='Share of targets receiving a pin',
            title='Independent-agent consensus coverage')
for ax in axes: ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(out_dir / 'integrity_stress_pin_shift.png', dpi=160)
plt.show()


## Interpretation

The detector sees the injected visits as ordinary input and recalculates every safeguard; it does not use a clean pin to penalise them. The clean pin and survey are used after that to measure shift and error. The independent-agent cluster method sharply reduces observed tail shifts where it emits a pin, but it abstains when independent corroboration is insufficient. The synthetic attacks are sensitivity tests, not real fraud labels or a production performance claim.
